In [ ]:
%sql
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.compras AS 
SELECT *
FROM READ_FILES('/Volumes/lakehouse_vertice/bronze/raw/compras/',
FORMAT => 'CSV',
HEADER => 'true',
INFERSCHEMA => 'true'
)

In [ ]:
%pip install openpyxl

import pandas as pd
import os

excel_dir = "/Volumes/lakehouse_vertice/bronze/raw/excel/"
xlsx_files = [f for f in os.listdir(excel_dir) if f.endswith(".xlsx")]

def sanitize_columns(cols):
    return [c.replace(' ', '_').replace('%', 'pct').replace('(', '').replace(')', '')
            .replace(',', '').replace(';', '').replace('{', '').replace('}', '')
            .replace('=', '').replace('\n', '').replace('\t', '') for c in cols]

all_pdfs = []
for fname in sorted(xlsx_files):
    pdf = pd.read_excel(os.path.join(excel_dir, fname))
    pdf.columns = sanitize_columns(pdf.columns)
    pdf["arquivo_origem"] = fname
    all_pdfs.append(pdf)
    print(f"{fname}: {len(pdf)} linhas, {len(pdf.columns)} colunas")

combined = pd.concat(all_pdfs, ignore_index=True)
print(f"\nTotal combinado: {len(combined)} linhas, {len(combined.columns)} colunas")

df = spark.createDataFrame(combined.astype(str))  # bronze: tudo como string
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("lakehouse_vertice.bronze.excel")
print("Tabela lakehouse_vertice.bronze.excel criada com sucesso!")

In [ ]:
%sql
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.financeiro AS
SELECT * 
FROM READ_FILES('/Volumes/lakehouse_vertice/bronze/raw/financeiro/',
    FORMAT =>'CSV',
    HEADER => TRUE,
    INFERSCHEMA => TRUE
)

In [ ]:
%sql

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.pmo as
SELECT *
FROM READ_FILES(
    '/Volumes/lakehouse_vertice/bronze/raw/pmo/',
    FORMAT => 'CSV',
    HEADER => TRUE,
    INFERSCHEMA => TRUE
)

In [ ]:
%sql

CREATE OR  REPLACE TABLE lakehouse_vertice.bronze.colaboradores AS
SELECT *
FROM READ_FILES('/Volumes/lakehouse_vertice/bronze/raw/rh/',
FORMAT => 'CSV',
HEADER => TRUE,
INFERSCHEMA => TRUE
)